# Twin experiments — pseudo-observations

Run SeapoPym at the 6 stations with the reference parameters of Table 1 and
store the resulting biomass as the *pseudo-observations* used by the genetic
algorithm (Manuscript Section 2.4.4).

The model is also run with `compute_initial_conditions=True` so we can
restart the GA-sampled simulations from the spin-up state without recomputing
it every time.

Inputs: `data/stations.zarr`, `parameters.yaml`.
Outputs: `data/pseudo_observations.zarr`, `data/initial_conditions.zarr`.
Runtime: ~30 s.

In [1]:
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import xarray as xr
import yaml
from seapopym.configuration.no_transport import (
    ForcingParameter,
    ForcingUnit,
    FunctionalGroupParameter,
    FunctionalGroupUnit,
    FunctionalTypeParameter,
    KernelParameter,
    MigratoryTypeParameter,
    NoTransportConfiguration,
)
from seapopym.model.no_transport_model import NoTransportModel


def _project_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Project root marker {marker!r} not found.")


PROJECT_ROOT = _project_root()
DATA_DIR = PROJECT_ROOT / "data"

with open(PROJECT_ROOT / "parameters.yaml") as f:
    PARAMS = yaml.safe_load(f)
REF = PARAMS["model_parameters"]["reference"]
GS = PARAMS["global_simulation"]
T_START = GS["start_date"]
T_ANALYSIS_START = (datetime.strptime(GS["spin_up_end"], "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
T_END = "2019-12-31"
print(f"Simulation: {T_START} -> {T_END} | pseudo-obs analysis window: {T_ANALYSIS_START} -> {T_END}")

Simulation: 1998-01-01 -> 2019-12-31 | pseudo-obs analysis window: 2000-01-01 -> 2019-12-31


## Build a (T, Z=1, Y=6, X=1) forcing grid from station time series

In [2]:
raw = xr.open_zarr(DATA_DIR / "stations.zarr").sel(time=slice(T_START, T_END)).load()
order = raw.station_lat.argsort().values
ordered = raw.isel(station=order)
y_values = ordered.station_lat.values.astype(np.float32)
x_values = np.array([0.0], dtype=np.float32)


def _to_grid(da: xr.DataArray) -> xr.DataArray:
    arr = da.values.T[:, :, np.newaxis]
    return xr.DataArray(
        arr,
        dims=("T", "Y", "X"),
        coords={"T": ordered.time.values, "Y": ("Y", y_values), "X": ("X", x_values)},
    )


temperature = _to_grid(ordered.temperature).expand_dims(Z=[0], axis=1)
npp = _to_grid(ordered.npp)
for coord, axis in {"T": "T", "Z": "Z", "Y": "Y", "X": "X"}.items():
    if coord in temperature.coords:
        temperature[coord].attrs["axis"] = axis
    if coord in npp.coords:
        npp[coord].attrs["axis"] = axis
temperature.attrs["units"] = "degC"
npp.attrs["units"] = "mg/m^2/day"

station_names_by_y = {float(lat): str(name) for lat, name in zip(y_values, ordered.station.values)}
station_names_by_y

{23.0: 'HOT',
 30.0: 'Canaries',
 32.0: 'BATS',
 45.5: 'Bay_of_Biscay',
 50.0: 'PAPA',
 75.0: 'BARENTS'}

## Run SeapoPym with the reference parameters

In [3]:
fg = FunctionalGroupParameter(functional_group=[FunctionalGroupUnit(
    name="zooplankton",
    energy_transfert=REF["energy_transfert"],
    functional_type=FunctionalTypeParameter(
        lambda_temperature_0=REF["lambda_temperature_0"],
        gamma_lambda_temperature=REF["gamma_lambda_temperature"],
        tr_0=REF["tr_0"],
        gamma_tr=REF["gamma_tr"],
    ),
    migratory_type=MigratoryTypeParameter(day_layer=0, night_layer=0),
)])

config = NoTransportConfiguration(
    forcing=ForcingParameter(
        temperature=ForcingUnit(forcing=temperature),
        primary_production=ForcingUnit(forcing=npp),
    ),
    functional_group=fg,
    kernel=KernelParameter(compute_initial_conditions=True),
)

with NoTransportModel.from_configuration(configuration=config) as model:
    model.run()
    model.state.compute()
    biomass = model.state.biomass.load().copy()
    initial_conditions = model.export_initial_conditions().load().copy()

None unit is milligram / day / meter ** 2, it will be converted to gram / day / meter ** 2.


None unit is milligram / day / meter ** 2, it will be converted to gram / day / meter ** 2.


## Persist the pseudo-observations and initial conditions

In [4]:
pseudo_obs_path = DATA_DIR / "pseudo_observations.zarr"
ic_path = DATA_DIR / "initial_conditions.zarr"

biomass_obs = biomass.sel(T=slice(T_ANALYSIS_START, T_END), functional_group=0, drop=True)
biomass_obs.to_dataset(name="observed_biomass").to_zarr(pseudo_obs_path, mode="w", zarr_format=2)

# Initial conditions have no T dimension after export; keep only what the GA needs.
initial_conditions = initial_conditions.drop_vars([v for v in ("T",) if v in initial_conditions.coords])
if "flag_values" in initial_conditions.functional_group.attrs:
    initial_conditions.functional_group.attrs["flag_values"] = str(initial_conditions.functional_group.attrs["flag_values"])
initial_conditions.to_zarr(ic_path, mode="w", zarr_format=2)

for p in (pseudo_obs_path, ic_path):
    size_mb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
    print(f"  {p.name}: {size_mb:.2f} MB")

  pseudo_observations.zarr: 0.31 MB
  initial_conditions.zarr: 0.01 MB
